In [ ]:
#packages and modules

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os

from dotenv import load_dotenv

env = load_dotenv('/mnt/d/linux/project/vars/.env')

In [ ]:
#variables

pgconn = os.getenv('POSTGRE_CONN')
pgdriver = os.getenv('PG_DRIVER')
pguser = os.getenv('POSTGRE_USER')
pgpwd = os.getenv('POSTGRE_PWD')
pgjar = os.getenv('POSTGRE_JAR')


In [ ]:
#creating session

spark = (( SparkSession
            .builder
                .appName('customers')
                .config('spark.jar', f'{pgjar}')
                .config('spark.driver.extraClassPath', f'{pgjar}')
                .config('spark.executor.extraClassPath', f'{pgjar}')
            .getOrCreate()
        ))

In [ ]:
#load customer data from source table

tmp_renamedrop_df = (( spark
                        .read
                        .format('jdbc')
                        .option('url', pgconn)
                        .option('driver', pgdriver)
                        .option('user', pguser)
                        .option('password', pgpwd)
                        .option('query', 
                                """ 
                                
                                    select

                                        customerid,
                                        customercategoryid,
                                        buyinggroupid,
                                        billtocustomerid,
                                        split_part(customername, '(', 1)as firstname,
		                                replace(split_part(customername, '(', 2), ')', '') as lastnameOrBranch,
                                        replace(split_part(customername, ',', 2),')', '') as place,
                                        accountopeneddate,
                                        creditlimit,
                                        standarddiscountpercentage,
                                        paymentdays,
                                        primarycontactpersonid,
                                        alternatecontactpersonid,
                                        deliverymethodid,
                                        deliverycityid,
                                        postalcityid,
                                        deliverypostalcode,
                                        deliveryaddressline1,
                                        deliveryaddressline2,
                                        postaladdressline1,
                                        postaladdressline2,
                                        isstatementsent,
                                        isoncredithold,
                                        _airbyte_ab_id,
                                        _airbyte_customers_hashid,
                                        _airbyte_emitted_at,
                                        _airbyte_normalized_at
                                        
                                    from sales.customers
                                
                                """)
                        
                    )).load().withColumnRenamed('_airbyte_ab_id', 'customer_elt_hashid')

In [ ]:
#load customercategory data from source table
tmp_custmercategory = (( spark
                            .read
                            .format('jdbc')
                            .option('url', pgconn)
                            .option('driver', pgdriver)
                            .option('user', pguser)
                            .option('password', pgpwd)
                            .option('query', 
                                    
                                    """ 
                                        select

                                            customercategoryid,
                                            customercategoryname,
                                            _airbyte_ab_id,
                                            _airbyte_customercategories_hashid
                                        
                                        from sales.customercategories
                                        
                                    """)
                        )).load().withColumnRenamed('_airbyte_ab_id', 'custcategory_elt_hashid')

In [ ]:
tmp_buyinggroups = (( spark
                        .read
                        .format('jdbc')
                        .option('url', pgconn)
                        .option('driver', pgdriver)
                        .option('user', pguser)
                        .option('password', pgpwd)
                        .option('query', 
                                
                                """
                                    select
                                    
                                        buyinggroupid,
                                        buyinggroupname,
                                        _airbyte_ab_id,
                                        _airbyte_buyinggroups_hashid
                                        
                                    from sales.buyinggroups
                                        
                                """)
                        )).load().withColumnRenamed('_airbyte_ab_id', 'custgroup_elt_hashid')

In [ ]:
#joining dataframes

joined_frames = (( tmp_renamedrop_df
                    .join(tmp_custmercategory, 'customercategoryid', 'left')
                    .join(tmp_buyinggroups, 'buyinggroupid', 'left')
                ))

In [ ]:
rename_df = ((
                joined_frames
                    .select('*', 
                            f.trim('firstname').alias('firstname_n'),
                            f.trim('lastnameOrbranch').alias('lastname'),
                            f.coalesce('creditlimit', f.lit(0.00)).alias('creditlimit_n'),
                            f.coalesce('buyinggroupid', f.lit(0)).alias('buyergroup_id'),
                            f.coalesce('alternatecontactpersonid', f.lit(0)).alias('alternatecontact_id')
                        )
))

In [232]:
renamedrop_df = ((rename_df
                     .drop('creditlimit')
                     .drop('buyinggroupid')
                     .drop('alternatecontactpersonid')
                     .drop('firstname')
                     .drop('lastnameOrBranch')
                     
                     #rename columns
                     .withColumnRenamed('creditlimit_n', 'creditlimit')
                     .withColumnRenamed('buyergroup_id', 'buyergroupid')
                     .withColumnRenamed('alternatecontact_id', 'alternatecontactid')
                     .withColumnRenamed('standarddiscountpercentage', 'discountpercentage')
                     .withColumnRenamed('primarycontactpersonid', 'primarycontactid')
                     .withColumnRenamed('deliverypostalcode', 'postalcode')
                     .withColumnRenamed('deliveryaddressline1', 'primaryaddress')
                     .withColumnRenamed('deliveryaddressline2', 'alternateaddress')
                     .withColumnRenamed('firstname_n', 'firstname')
                     .withColumnRenamed('postaladdressline1', 'postaladdress')
                     .withColumnRenamed('postaladdressline2', 'alternatepostaladdress')
                     .withColumnRenamed('_airbyte_customers_hashid', 'customers_hashid')
                     .withColumnRenamed('customer_elt_hashid', 'elt_customer_hashid')
                     .withColumnRenamed('_airbyte_customercategories_hashid', 'customercategory_hashid')
                     .withColumnRenamed('custcategory_elt_hashid', 'elt_custcategory_hashid')
                     .withColumnRenamed('_airbyte_buyinggroups_hashid', 'elt_buyergroup_hashid')
                     .withColumnRenamed('custgroup_elt_hashid', 'buyergroup_hashid')
                     .withColumnRenamed('_airbyte_emitted_at', 'elt_start')
                     .withColumnRenamed('_airbyte_normalized_at', 'elt_finalized')
))

In [233]:
df = renamedrop_df.withColumn('lastname', f.split('lastname', '[^A-Za-z0-9 '']'))

renamedrop_df = df.withColumn('lastname', df['lastname'].getItem(0))



In [238]:
#organizing columns

customerframe= ((
                renamedrop_df
                    .select('customerid',
                            'customercategoryid',
                            'billtocustomerid',
                            'buyergroupid',
                            'firstname',
                            'lastname',
                            'customercategoryname',
                            'buyinggroupname',
                            'accountopeneddate',
                            'paymentdays',
                            'creditlimit',
                            'discountpercentage',
                            'primarycontactid',
                            'alternatecontactid',
                            'deliverymethodid',
                            'deliverycityid',
                            'postalcityid',
                            'postalcode',
                            'primaryaddress',
                            'alternateaddress',
                            'postaladdress',
                            'alternatepostaladdress',
                            'place',
                            'isoncredithold',
                            'isstatementsent',
                            'elt_customer_hashid',
                            'customers_hashid',
                            'elt_custcategory_hashid',
                            'customercategory_hashid',
                            'elt_buyergroup_hashid',
                            'buyergroup_hashid'
                            )
                    
                    #change boolean for 'binary'
                    .withColumn('isoncredithold', f.when(renamedrop_df['isoncredithold'] == 'true', 1).otherwise(0))
                    .withColumn('isstatementsent', f.when(renamedrop_df['isstatementsent'] == 'true', 1).otherwise(0))
                #     .withColumn('place', f.when(renamedrop_df['lastname'] == 'Head Office', 'Head Qt').otherwise('place'))
        
)).show()

+----------+------------------+----------------+------------+-------------+--------------+--------------------+---------------+-----------------+-----------+-----------+------------------+----------------+------------------+----------------+--------------+------------+----------+--------------+--------------------+-------------+----------------------+-----+--------------+---------------+--------------------+--------------------+-----------------------+-----------------------+---------------------+--------------------+
|customerid|customercategoryid|billtocustomerid|buyergroupid|    firstname|      lastname|customercategoryname|buyinggroupname|accountopeneddate|paymentdays|creditlimit|discountpercentage|primarycontactid|alternatecontactid|deliverymethodid|deliverycityid|postalcityid|postalcode|primaryaddress|    alternateaddress|postaladdress|alternatepostaladdress|place|isoncredithold|isstatementsent| elt_customer_hashid|    customers_hashid|elt_custcategory_hashid|customercategory_has